# Tutorial 2: Understanding and Using CycleReviewer

## Introduction

CycleReviewer (also known as WhizReviewer) is a set of specialized large language models that have undergone extensive supervised training specifically for academic peer review. The model is available in three sizes:

- 8B parameters (based on Llama3.1)
- 70B parameters (based on Llama3.1)
- 123B parameters (based on Mistral-Large-2)

These models are designed to simulate the peer review process by evaluating research papers across multiple dimensions, providing detailed feedback, and generating comprehensive reviews that closely mirror those produced by human academic reviewers.

## Setting Up the Environment

Let's start by importing the necessary libraries and initializing the CycleReviewer model:

In [1]:
# 设置使用 GPU 1（因为 GPU 0 已被 CycleResearcher 占用）
# import os
# os.environ['CUDA_VISIBLE_DEVICES'] = '1'

# print(f"Using GPU: {os.environ.get('CUDA_VISIBLE_DEVICES')}")

In [ ]:
import json

from ai_researcher import CycleReviewer

# Initialize CycleReviewer with the 8B parameter model (for faster inference)
# You can choose "8B", "70B", or "123B" depending on available computational resources
reviewer = CycleReviewer(model_size="8B")

INFO 01-27 14:57:07 [utils.py:263] non-default args: {'max_model_len': 50000, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'model': 'WestlakeNLP/CycleReviewer-ML-Llama-3.1-8B'}
INFO 01-27 14:57:08 [model.py:530] Resolved architecture: LlamaForCausalLM
INFO 01-27 14:57:08 [model.py:1545] Using max model len 50000
INFO 01-27 14:57:09 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 01-27 14:57:09 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 01-27 14:57:09 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduling.
(EngineCore_DP0 pid=1995302) INFO 01-27 14:57:12 [core.py:97] Initializing a V1 LLM engine (v0.14.0) with config: model='WestlakeNLP/CycleReviewer-ML-Llama-3.1-8B', speculative_config=None, tokenizer='WestlakeNLP/CycleReviewer-ML-Llama-3.1-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=50000, do

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore_DP0 pid=1995302) INFO 01-27 14:57:40 [default_loader.py:291] Loading weights took 22.94 seconds
(EngineCore_DP0 pid=1995302) INFO 01-27 14:57:41 [gpu_model_runner.py:3905] Model loading took 14.99 GiB memory and 25.145680 seconds
(EngineCore_DP0 pid=1995302) INFO 01-27 14:57:46 [backends.py:644] Using cache directory: /home/zhihang/.cache/vllm/torch_compile_cache/8fde41407a/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1995302) INFO 01-27 14:57:46 [backends.py:704] Dynamo bytecode transform time: 4.19 s
(EngineCore_DP0 pid=1995302) INFO 01-27 14:57:49 [backends.py:226] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.263 s
(EngineCore_DP0 pid=1995302) INFO 01-27 14:57:49 [monitor.py:34] torch.compile takes 5.45 s in total
(EngineCore_DP0 pid=1995302) INFO 01-27 14:57:49 [gpu_worker.py:358] Available KV cache memory: 13.48 GiB
(EngineCore_DP0 pid=1995302) INFO 01-27 14:57:50 [kv_cache_utils.py:1305] GPU KV cache size: 11

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 24.53it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 27.52it/s]


(EngineCore_DP0 pid=1995302) INFO 01-27 14:57:54 [gpu_model_runner.py:4856] Graph capturing finished in 4 secs, took 0.37 GiB
(EngineCore_DP0 pid=1995302) INFO 01-27 14:57:54 [core.py:273] init engine (profile, create kv cache, warmup model) took 12.73 seconds
INFO 01-27 14:57:54 [llm.py:347] Supported tasks: ['generate']


## Loading a Paper to Review

Let's load a paper from a JSON file and prepare it for review:

In [3]:
# Load a paper from our JSON file
with open('generated_paper.json', 'r', encoding='utf-8') as f:
    papers = json.load(f)

# Print basic information about the paper
print(f"Paper Title: {papers[0]['title']}")
print(f"Abstract length: {len(papers[0]['abstract'])} characters")
print(f"LaTeX content length: {len(papers[0]['latex'])} characters")

Paper Title: Improving AI Scientists through Multi-Agent Competitive Preference Optimization


Abstract length: 1182 characters
LaTeX content length: 45723 characters


## Reviewing a Single Paper

Now, let's use CycleReviewer to evaluate the paper:

In [4]:
# Generate the review
review_result = reviewer.evaluate([paper['latex'] for paper in papers])

Adding requests:   0%|          | 0/9 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

## Parsing and Analyzing Review Results

Let's parse the review to extract structured information including ratings, strengths, weaknesses, and recommendations:

In [5]:
# Parse the review
review_data = review_result[0]

for i in range(len(review_result)):
    if review_result[i] != None:
        review_data = review_result[i]
        # Print summary of the review
        print(f"Title: {papers[i]['title']}")
        print(f"Average Rating: {review_data['avg_rating']:.2f}/10")
        print(f"Decision: {review_data['paper_decision']}")
        print(f"Number of Reviewers: {len(review_data['reviews'])}")
        print("\nIndividual Ratings:")
        for i, rating in enumerate(review_data['rating']):
            print(f"  Reviewer {i+1}: {rating}")
        print('---***---')

Title: Improving AI Scientists through Multi-Agent Competitive Preference Optimization


Average Rating: 5.75/10
Decision: Reject
Number of Reviewers: 4

Individual Ratings:
  Reviewer 1: 5.0
  Reviewer 2: 6.0
  Reviewer 3: 6.0
  Reviewer 4: 6.0
---***---
Title: ReviewerNet: Harnessing Multi-Agent Systems for Comprehensive Scientific Paper Review


Average Rating: 4.50/10
Decision: Reject
Number of Reviewers: 4

Individual Ratings:
  Reviewer 1: 3.0
  Reviewer 2: 5.0
  Reviewer 3: 5.0
  Reviewer 4: 5.0
---***---
Title: Reviewer Agent: Enhancing Scientific Peer Review Experience with Reviewer Agents


Average Rating: 0.00/10
Decision: Reject
Number of Reviewers: 4

Individual Ratings:
  Reviewer 1: 0
  Reviewer 2: 0
  Reviewer 3: 0
  Reviewer 4: 0
---***---
Title: Scientific Review Agents: A Step Towards Artificial Intelligence-Assisted Peers


Average Rating: 0.00/10
Decision: Reject
Number of Reviewers: 1

Individual Ratings:
  Reviewer 1: 0
---***---
Title: Towards Autonomous Scienti

## Let's examine some key feedback from the reviewers:

In [6]:
# Display key strengths mentioned by reviewers
print("KEY STRENGTHS IDENTIFIED:")
print("-" * 50)
for i, strength in enumerate(review_data['strength']):
    print(f"Reviewer {i+1}:")
    # Print the first 200 characters of each strength for brevity
    print(strength[:500] + "..." if len(strength) > 200 else strength)
    print()

# Display key weaknesses mentioned by reviewers
print("\nKEY WEAKNESSES IDENTIFIED:")
print("-" * 50)
for i, weakness in enumerate(review_data['weaknesses']):
    print(f"Reviewer {i+1}:")
    # Print the first 200 characters of each weakness for brevity
    print(weakness[:500] + "..." if len(weakness) > 200 else weakness)
    print()

KEY STRENGTHS IDENTIFIED:
--------------------------------------------------
Reviewer 1:
- The paper is well-written and easy to follow.

- The authors conducted a comprehensive quality analysis of reviews generated by human and LLM-assisted reviewers through three dimensions: general quality assessment, recommendation quality, and review quality.

...

Reviewer 2:
- The paper is well-written and easy to follow.

- The authors conducted a comprehensive quality analysis of reviews generated by human and LLM-assisted reviewers through three dimensions: general quality assessment, recommendation quality, and review quality.

...

Reviewer 3:
The paper is well-written and easy to follow. The authors have conducted a comprehensive quality analysis of reviews generated by human and LLM-assisted reviewers through three dimensions: general quality assessment, recommendation quality, and review quality.

...

Reviewer 4:
- The paper is well-written and easy to follow.

- The authors conducted a

### Meta Review

Let's look at the meta review, which synthesizes the individual reviewer feedback:

In [7]:
print("META REVIEW:")
print("=" * 50)
print(review_data['meta_review'])

META REVIEW:
This paper proposes an AI-assisted reviewer LLM that can perform full peer review of scientific manuscripts through role-playing and a multi-round iterative refinement approach. They fine-tune reward models, applied iterative preference optimization, and chain-of-though optimizations to optimize their method. To assess its effectiveness and efficiency, they conducted a comprehensive quality analysis of reviews generated by human and LLM-assisted reviewers through three dimensions: general quality assessment, recommendation quality, and review quality. Extensive results show that their method outperforms human reviews and OpenAI GPT-4 Turbo in general and quality assessment, outperforms Author-ASSIST-revised in recommendations, and performs competitively with Author-ASSIST and Author-ASSIST-revisd in review quality, demonstrating the effectiveness and efficiency of their approach.

The paper has received 4 reviews, all of which are negative. The authors have not responded t

## Finding the Best Paper

Now, let's find the highest-rated paper from our reviews:

In [8]:
# Find the best paper
rating_max = 0
best_paper_num = 0
for i in range(len(review_result)):
    if review_result[i] != None:
        ratings = review_data['avg_rating']
        if ratings > rating_max:
            best_paper_num=i
            rating_max = ratings


print("=" * 50)
print("BEST PAPER SELECTED:")
print(f"Title: {papers[best_paper_num]['title']}...")
print(f"Average Rating: {review_result[best_paper_num]['avg_rating']:.2f}/10")
print(f"Decision: {review_result[best_paper_num]['paper_decision']}")
print("=" * 50)

# Print key strengths from the highest-rated paper's review
best_review = review_result[best_paper_num]
print("\nKEY STRENGTHS OF THE BEST PAPER:")
for i, strength in enumerate(best_review.get('strength', [])):
    if strength:
        print(f"\nStrength {i+1}:")
        print(strength[:300] + "..." if len(strength) > 300 else strength)

BEST PAPER SELECTED:
Title: Improving AI Scientists through Multi-Agent Competitive Preference Optimization

...
Average Rating: 5.75/10
Decision: Reject

KEY STRENGTHS OF THE BEST PAPER:

Strength 1:
- The paper is well-written and easy to follow.
- The proposed method is novel and interesting.



Strength 2:
The paper is well-written and easy to follow. The authors provide a clear description of their method and its evaluation. The proposed method is novel and interesting, and the results are promising.



Strength 3:
1. The paper is well-written and easy to follow.
2. The authors introduce a novel multi-agent competitive optimization approach that combines diverse model dynamics and multi-agent reinforcement learning.
3. The experiments show that the proposed method outperforms previous methods in terms of the q...

Strength 4:
1. The paper is well-written and easy to follow.
2. The authors introduce a novel multi-agent competitive optimization approach that combines diverse model d

## Conclusion

In this tutorial, we've explored how to use CycleReviewer to evaluate academic research papers. We've seen how the model can:

1. Generate detailed reviews that assess papers on multiple dimensions
2. Provide specific feedback on strengths and weaknesses
3. Assign numerical ratings to quantify paper quality
4. Generate meta-reviews that synthesize multiple reviewer perspectives
5. Make accept/reject recommendations

CycleReviewer represents a significant advancement in automating the peer review process, offering researchers, publishers, and educators a powerful tool for evaluating scientific work. While it can provide valuable feedback and insights, it's important to remember that these automated reviews are best used in conjunction with human expert evaluation, especially for final publication decisions.